<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/PythonGPTDecoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets

In [ ]:
!pip install transformers

In [ ]:
from datasets import load_dataset
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModelForCausalLM,AutoModel,AutoTokenizer
from torch.utils.data import DataLoader,Dataset
import pandas as pd
import gc
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [ ]:
hf_data = load_dataset("sahil2801/CodeAlpaca-20k")

In [ ]:
print(hf_data)

DatasetDict({
    train: Dataset({
        features: ['output', 'instruction', 'input'],
        num_rows: 20022
    })
})


In [ ]:
df = pd.DataFrame(hf_data['train'])
df.head(5)

,output,instruction,input
0,"arr = [2, 4, 6, 8, 10]",Create an array of length 5 which contains all...,
1,Height of triangle = opposite side length * si...,Formulate an equation to calculate the height ...,
2,"def replace(self, replace_with):\n new_stri...",Write a replace method for a string class whic...,"string = ""Hello World!""\nreplace_with = ""Greet..."
3,"arr = [3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33...",Create an array of length 15 containing number...,
4,def find_num_distinct_states(matrix):\n sta...,Write a function to find the number of distinc...,"matrix = [[1, 0, 0],\n [1, 0, 1],\n ..."


In [ ]:
def combine_columns(row):
  instruction = str(row['instruction']).strip()
  extra_input = str(row['input']).strip()
  output_code = str(row['output']).strip()

  if extra_input and extra_input.lower() != 'nan' and extra_input != "":
    return f"Instruction: {instruction}\nContext: {extra_input}\n### Code:\n{output_code}"
  else:
    return f"Instruction: {instruction}\n### Code:\n{output_code}"

df['text'] = df.apply(combine_columns,axis=1)
print(df['text'].iloc[1])

Instruction: Formulate an equation to calculate the height of a triangle given the angle, side lengths and opposite side length.
### Code:
Height of triangle = opposite side length * sin (angle) / side length


In [ ]:
autoToken = AutoTokenizer.from_pretrained('gpt2')
autoToken.pad_token = autoToken.eos_token

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
max_len = 256
vocab_size = autoToken.vocab_size

In [ ]:
statement_text = df['text'].astype(str).values

In [ ]:
x_train,x_test = train_test_split(statement_text,test_size=0.3,random_state=42)

In [ ]:
train_tokeniser = autoToken(text=list(x_train),padding='max_length',max_length=max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')
val_tokeniser = autoToken(text=list(x_test),padding='max_length',max_length=max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

In [ ]:
class PythonGPT(Dataset):
  def __init__(self,encoding):
    self.encoding = encoding

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    input_ids = self.encoding['input_ids'][idx]
    mask = self.encoding['attention_mask'][idx]

    return {
        'input_ids':input_ids[:-1],
        'attention_mask':mask[:-1],
        'target_ids':input_ids[1:]
    }

In [ ]:
train_ds = PythonGPT(train_tokeniser)
test_ds = PythonGPT(val_tokeniser)

In [ ]:
train_loader = DataLoader(dataset=train_ds,batch_size=16,shuffle=True,pin_memory=True,num_workers=2)
val_loader = DataLoader(dataset=test_ds,batch_size=16,shuffle=False,pin_memory=True,num_workers=2)

In [ ]:
class GPTWritter(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.gpt2 = AutoModel.from_pretrained('gpt2')
    self.dropout = nn.Dropout(0.2)
    self.output_layer = nn.Linear(768,vocab_size)

  def forward(self,input_ids,mask):
    outputs = self.gpt2(input_ids=input_ids,attention_mask=mask)
    x = outputs.last_hidden_state
    x = self.dropout(x)
    return self.output_layer(x)

In [ ]:
model = GPTWritter(vocab_size=vocab_size)

if torch.cuda.device_count() > 1:
  model = nn.DataParallel(model)
model = model.to(device)

In [ ]:
optimizer = optim.AdamW(model.parameters(),lr=5e-5)
loss_fn = nn.CrossEntropyLoss()
scalar = torch.cuda.amp.GradScaler()

/tmp/ipykernel_57/2312945097.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scalar = torch.cuda.amp.GradScaler()


In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
epochs = 4
for epoch in range(epochs):
  model.train()
  train_loss = 0
  progress_bar_train = tqdm(train_loader,desc=f"Epoch {epoch+1}")

  for batch in progress_bar_train:
    input_ids = batch['input_ids'].to(device).squeeze(1)
    attention_mask = batch['attention_mask'].to(device).squeeze(1)
    target_ids = batch['target_ids'].to(device).squeeze(1)

    optimizer.zero_grad()

    with torch.cuda.amp.autocast():
      outputs = model(input_ids,attention_mask)
      loss = loss_fn(outputs.view(-1,outputs.size(-1)),target_ids.view(-1))

    scalar.scale(loss).backward()
    scalar.step(optimizer)
    scalar.update()

    train_loss += loss.item()
    progress_bar_train.set_postfix(loss=loss.item())

  training_loss = train_loss / len(train_loader)
  training_perplexity = torch.exp(torch.tensor(training_loss))

  print(f"Train_loss: {training_loss:.4f} | Train_perplexity: {training_perplexity:.2f}")

  #Validation
  model.eval()
  average_val_loss = 0
  progressbar_val = tqdm(val_loader,desc=f"Validation:")
  with torch.no_grad():
    for batch in progressbar_val:
      val_input_ids = batch['input_ids'].to(device).squeeze(1)
      val_attention_mask = batch['attention_mask'].to(device).squeeze(1)
      val_target_ids = batch['target_ids'].to(device).squeeze(1)

      with torch.cuda.amp.autocast():
        val_output = model(val_input_ids,val_attention_mask)
        val_losss = loss_fn(val_output.view(-1,val_output.size(-1)),val_target_ids.view(-1))

      average_val_loss += val_losss.item()
      progressbar_val.set_postfix(loss=val_losss.item())

  validation_loss = average_val_loss / len(val_loader)
  val_perplaxity = torch.exp(torch.tensor(validation_loss))

  print(f"Epoch: {epoch+1} | Val_loss: {validation_loss:.4f} | Val_perplexity: {val_perplaxity:.2f}")

  gc.collect()
  torch.cuda.empty_cache()


Epoch 1:   0%|          | 0/876 [00:00<?, ?it/s]/tmp/ipykernel_57/296076208.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1: 100%|██████████| 876/876 [06:23<00:00,  2.29it/s, loss=0.807]


Train_loss: 0.5944 | Train_perplexity: 1.81


Validation::   0%|          | 0/376 [00:00<?, ?it/s]/tmp/ipykernel_57/296076208.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Validation:: 100%|██████████| 376/376 [01:03<00:00,  5.97it/s, loss=0.553]


Epoch: 1 | Val_loss: 0.5729 | Val_perplexity: 1.77


Epoch 2: 100%|██████████| 876/876 [06:21<00:00,  2.29it/s, loss=0.442]


...

Epoch: 3 | Val_loss: 0.5270 | Val_perplexity: 1.69


Epoch 4: 100%|██████████| 876/876 [06:21<00:00,  2.29it/s, loss=0.436]


Train_loss: 0.4697 | Train_perplexity: 1.60


Validation:: 100%|██████████| 376/376 [01:03<00:00,  5.96it/s, loss=0.507]


Epoch: 4 | Val_loss: 0.5129 | Val_perplexity: 1.67


In [ ]:
def generate_news(model, tokenizer, prompt, max_new_tokens=50):
    model.eval()

    # 1. Turn your headline prompt into token numbers
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    # Extract the true inner model if DataParallel is active
    actual_model = model.module if isinstance(model, nn.DataParallel) else model

    for _ in range(max_new_tokens):
        with torch.no_grad():
            with torch.cuda.amp.autocast():
                # 2. Get the vocabulary predictions matrix
                logits = actual_model(input_ids, attention_mask=None)

        # 3. Focus entirely on the predictions for the VERY LAST word slot
        next_token_logits = logits[:, -1, :]

        # 4. Select the highest scoring word (Classification winner!)
        next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)

        # 5. Concatenate the new word token onto the sequence string array
        input_ids = torch.cat([input_ids, next_token], dim=-1)

        # Stop typing if the model hits the End-of-Text boundary token
        if next_token.item() == tokenizer.eos_token_id:
            break

    # 6. Translate the raw numbers back into readable English text
    return tokenizer.decode(input_ids, skip_special_tokens=True)[0]

# 💥 RUN THESE EXAMPLES IN A NEW CELL TO SEE IT WRITE
print("--- TEST 1 ---")
my_coding_prompt = "Instruction: Write a Python function to check if a number is even.\n### Code:\n"
print(generate_news(model, autoToken, my_coding_prompt))

--- TEST 1 ---


NameError: name 'model' is not defined